# Avaliação do PTT5-small + LoRA no conjunto de teste

TCC — tradução de português para glosa em Libras. Este caderno usa o adaptador já treinado em `MyDrive/libras-tcc/models/lora_adapter` e o `test.csv` em `MyDrive/libras-tcc/data`.

**Como usar:** Ambiente de execução → Alterar tipo → GPU (T4) → Executar tudo. Leva poucos minutos. No final, copie o bloco entre `=== INÍCIO ===` e `=== FIM ===` e cole no chat.

In [ ]:
!pip install -q transformers peft accelerate sentencepiece sacrebleu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/libras-tcc'
ADAPTER_DIR = f'{PROJECT_DIR}/models/lora_adapter'
import os
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
assert os.path.exists(f'{ADAPTER_DIR}/adapter_model.safetensors'), 'adaptador não encontrado no Drive'
assert os.path.exists(f'{PROJECT_DIR}/data/test.csv'), 'test.csv não encontrado no Drive'

In [ ]:
import torch, pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = 'unicamp-dl/ptt5-small-portuguese-vocab'
TASK_PREFIX = 'traduzir para glosa em libras: '
MAX_SOURCE_LEN = MAX_TARGET_LEN = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
model = PeftModel.from_pretrained(AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME), ADAPTER_DIR).to(device).eval()
test_df = pd.read_csv(f'{PROJECT_DIR}/data/test.csv')
print(len(test_df), 'pares de teste |', device)

In [ ]:
from tqdm.auto import tqdm

# mesma decodificação do treino: busca em feixe com largura 4
texts = [TASK_PREFIX + str(x) for x in test_df['pt']]
order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
preds = [None] * len(texts)
BATCH = 64
with torch.no_grad():
    for s in tqdm(range(0, len(order), BATCH)):
        idx = order[s:s + BATCH]
        enc = tokenizer([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=MAX_SOURCE_LEN).to(device)
        out = model.generate(**enc, max_length=MAX_TARGET_LEN, num_beams=4)
        for i, o in zip(idx, tokenizer.batch_decode(out, skip_special_tokens=True)):
            preds[i] = o
test_df['glosa_pred_lora'] = preds
test_df.to_csv(f'{PROJECT_DIR}/results/predictions_lora.csv', index=False)
print('Predições salvas em results/predictions_lora.csv')

In [ ]:
import re, sacrebleu

# Mesma normalização e mesmas categorias de text_utils.py / evaluate.py
def normalize_glosa(text):
    if not isinstance(text, str):
        return ''
    s = text.strip().upper()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'\s*\[\s*', ' [', s)
    s = re.sub(r'\s*\]\s*', '] ', s)
    return re.sub(r'\s+', ' ', s).strip()

CATEGORIES = {
    'marcador_ponto': r'\[PONTO\]',
    'desambiguacao_amp': r'&',
    'composto_underscore': r'(?<![123][SP])_(?!\d)',
    'numeral': r'\d',
    'negacao_fundida': r'\bNÃO_',
    'interrogacao': r'\[INTERROGAÇÃO\]',
    'verbo_direcional': r'\b[123][SP]_',
    'negacao_isolada': r'\bNÃO\b(?!_)',
    'exclamacao': r'\[EXCLAMAÇÃO\]',
}

refs = [normalize_glosa(x) for x in test_df['glosa'].astype(str)]
hyps = [normalize_glosa(x) for x in test_df['glosa_pred_lora'].fillna('').astype(str)]

def score(mask=None):
    r = [x for x, m in zip(refs, mask) if m] if mask is not None else refs
    h = [x for x, m in zip(hyps, mask) if m] if mask is not None else hyps
    return len(r), round(sacrebleu.corpus_bleu(h, [r]).score, 2), round(sacrebleu.corpus_chrf(h, [r]).score, 2)

lines = ['=== INÍCIO ===']
n, b, c = score()
lines.append(f'GLOBAL n={n} BLEU={b} chrF={c}')
for name, rx in CATEGORIES.items():
    mask = [bool(re.search(rx, x)) for x in refs]
    n, b, c = score(mask)
    lines.append(f'CATEGORIA {name} n={n} BLEU={b} chrF={c}')
for label, val in (('Geral', False), ('Governamental', True)):
    mask = (test_df['is_government_source'].astype(str).str.lower() == str(val).lower()).tolist()
    n, b, c = score(mask)
    lines.append(f'DOMINIO {label} n={n} BLEU={b} chrF={c}')
vazias = sum(1 for h in hyps if not h)
rep = sum(1 for h in hyps if re.search(r'\b(\S+)( \1){2,}\b', h))
lines.append(f'DIAGNOSTICO saidas_vazias={vazias} saidas_com_repeticao_3x={rep}')
lines.append('EXEMPLOS')
for _, r in test_df.sample(12, random_state=42).iterrows():
    lines.append(f"PT: {r['pt']}\nREF: {normalize_glosa(r['glosa'])}\nLORA: {normalize_glosa(r['glosa_pred_lora'])}\n")
lines.append('=== FIM ===')
report = '\n'.join(lines)
open(f'{PROJECT_DIR}/results/avaliacao_lora.txt', 'w').write(report)
print(report)